## 1. Environment Setup and Library Imports

In [ ]:
import os
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

# -------------------------------------------------------------------------
# Path resolution to access root directory and 'src' modules
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Ensure root directory is registered in sys.path for custom module imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import local custom modules
from src.pkldataset import PKLDataset, class_to_onehot
from src.helpers import set_seed, get_model, train_model, eval_model

# Hardware configuration and loss function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# Random seeds for evaluation (10 seeds matching RP and AZ benchmarks)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# Sheet thickness and baseline domain configuration
source_thickness = "T197"
target_thickness = "T197"

# -------------------------------------------------------------------------
# Stratified Sampling Helper (Supports both PKLDataset and Subset objects)
# -------------------------------------------------------------------------
def get_balanced_transfer_subset(dataset, total_samples, seed=42):
    """
    Extracts an exactly class-balanced subset from a PKLDataset or Subset instance.
    For example: 20 samples -> 2 samples per wear class (R05..R50).
    """
    np.random.seed(seed)
    num_classes = 10
    samples_per_class = total_samples // num_classes
    
    # Handle both full PKLDataset instances and Subset splits
    if isinstance(dataset, Subset):
        base_dataset = dataset.dataset
        available_indices = dataset.indices
    else:
        base_dataset = dataset
        available_indices = list(range(len(dataset.files)))
    
    # Map subset indices by wear class (R05 to R50)
    class_indices = {i: [] for i in range(num_classes)}
    for idx in available_indices:
        filename = base_dataset.files[idx]
        class_label = filename.split('_')[0]
        c_idx = int(np.argmax(class_to_onehot[class_label]))
        class_indices[c_idx].append(idx)
        
    selected_indices = []
    for c_idx, indices in class_indices.items():
        chosen = np.random.choice(indices, size=samples_per_class, replace=False)
        selected_indices.extend(chosen)
        
    return Subset(base_dataset, selected_indices)

# -------------------------------------------------------------------------
# Dataset Paths & Configurations (Multimodal Fusion: RP + AZ, 2 channels)
# -------------------------------------------------------------------------
# In fusion mode, PKLDataset is pointed to the RP folder and pairs it with AZ automatically
train_path_1 = str(DATA_DIR / "HC" / source_thickness / "RP")
target_train_path = str(DATA_DIR / "DC" / target_thickness / "RP")

# Transfer learning sample sizes (simulating train_20, train_50, ..., train_500)
transfer_sample_sizes = [20, 50, 100, 200, 300, 400, 500]
transfer_sets = [f"train_{size}" for size in transfer_sample_sizes]

# Validation sets across all DC sheet thicknesses (held-out evaluation sets for Fusion)
val_paths = [
    str(DATA_DIR / "DC" / "T185" / "RP"),
    str(DATA_DIR / "DC" / "T188" / "RP"),
    str(DATA_DIR / "DC" / "T191" / "RP"),
    str(DATA_DIR / "DC" / "T194" / "RP"),
    str(DATA_DIR / "DC" / "T197" / "RP"),
]

# -------------------------------------------------------------------------
# Strict 70/30 Split on Target Domain (DC / T197 / Fusion) to eliminate leakage
# -------------------------------------------------------------------------
# 70% pool for fine-tuning subsets (3,500 multimodal samples)
# 30% strictly held-out test set for T197 evaluation (1,500 multimodal samples)
train_pool_dc, test_ds_t197 = PKLDataset.split_dataset(
    folder_path=target_train_path,
    modality="fusion",
    train_ratio=0.7,
    random_seed=42
)

# Prepare results container matching the reference format: results[t][vp] -> []
results = {t: {vp: [] for vp in val_paths} for t in transfer_sets}

# Verify folder paths existence
print("\nVerifying folder paths for Multimodal Fusion (RP + AZ)...")
for p in [train_path_1, target_train_path] + val_paths:
    exists = os.path.exists(p)
    print(f"Path ({p}): {'FOUND' if exists else 'MISSING'}")

print(f"\nHC Source Pretraining Path (Fusion): {train_path_1}")
print(f"DC Target Fine-Tuning Path (Fusion): {target_train_path}")
print(f"DC Transfer Sets to Evaluate: {transfer_sets}")
print(f"DC Validation Thickness Paths (Fusion): {val_paths}")
print(f"DC T197 Train Pool Size (70%): {len(train_pool_dc)} samples")
print(f"DC T197 Held-Out Test Set Size (30%): {len(test_ds_t197)} samples")
print(f"Total Random Seeds: {len(seeds)}")

## 2. Configuration (Fusion)

In [ ]:
import sys
from pathlib import Path
import numpy as np
from torch.utils.data import Subset

# -------------------------------------------------------------------------
# Path resolution
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pkldataset import PKLDataset, class_to_onehot

# -------------------------------------------------------------------------
# Stratified Sampling Helper (Supports both PKLDataset and Subset objects)
# -------------------------------------------------------------------------
def get_balanced_transfer_subset(dataset, total_samples, seed=42):
    """
    Extracts an exactly class-balanced subset from a PKLDataset or Subset instance.
    For example: 20 samples -> 2 samples per wear class (R05..R50).
    """
    np.random.seed(seed)
    num_classes = 10
    samples_per_class = total_samples // num_classes
    
    # Handle both full PKLDataset instances and Subset splits
    if isinstance(dataset, Subset):
        base_dataset = dataset.dataset
        available_indices = dataset.indices
    else:
        base_dataset = dataset
        available_indices = list(range(len(dataset.files)))
    
    # Map subset indices by wear class (R05 to R50)
    class_indices = {i: [] for i in range(num_classes)}
    for idx in available_indices:
        filename = base_dataset.files[idx]
        class_label = filename.split('_')[0]
        c_idx = int(np.argmax(class_to_onehot[class_label]))
        class_indices[c_idx].append(idx)
        
    selected_indices = []
    for c_idx, indices in class_indices.items():
        chosen = np.random.choice(indices, size=samples_per_class, replace=False)
        selected_indices.extend(chosen)
        
    return Subset(base_dataset, selected_indices)

# -------------------------------------------------------------------------
# Dataset Paths (Multimodal Fusion: RP base path, pairs resolved to AZ)
# -------------------------------------------------------------------------
# Primary pretraining path: HC source domain (T197 base path)
train_path_1 = str(DATA_DIR / "HC" / "T197" / "RP")

# Target domain path for fine-tuning: DC base folder
target_train_path = str(DATA_DIR / "DC" / "T197" / "RP")

# Transfer learning sample sizes (simulating train_20, train_50, ..., train_500)
transfer_sample_sizes = [20, 50, 100, 200, 300, 400, 500]
transfer_sets = [f"train_{size}" for size in transfer_sample_sizes]

# Validation sets across all DC sheet thicknesses (held-out evaluation sets)
val_paths = [
    str(DATA_DIR / "DC" / "T185" / "RP"),
    str(DATA_DIR / "DC" / "T188" / "RP"),
    str(DATA_DIR / "DC" / "T191" / "RP"),
    str(DATA_DIR / "DC" / "T194" / "RP"),
    str(DATA_DIR / "DC" / "T197" / "RP"),
]

# Random seeds for reproducibility across runs
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# -------------------------------------------------------------------------
# Strict 70/30 Split on Target Domain (DC / T197 / Fusion) to eliminate leakage
# -------------------------------------------------------------------------
# 70% pool for fine-tuning subsets (3,500 multimodal samples)
# 30% strictly held-out test set for T197 evaluation (1,500 multimodal samples)
train_pool_dc, test_ds_t197 = PKLDataset.split_dataset(
    folder_path=target_train_path,
    modality="fusion",
    train_ratio=0.7,
    random_seed=42
)

# Prepare results container matching the reference format: results[t][vp] -> []
results = {t: {vp: [] for vp in val_paths} for t in transfer_sets}

print(f"HC Source Pretraining Base Path (Fusion): {train_path_1}")
print(f"DC Target Fine-Tuning Base Path (Fusion): {target_train_path}")
print(f"DC Transfer Sets to Evaluate: {transfer_sets}")
print(f"DC Validation Thickness Paths (Fusion): {val_paths}")
print(f"DC T197 Train Pool Size (70%): {len(train_pool_dc)} samples")
print(f"DC T197 Held-Out Test Set Size (30%): {len(test_ds_t197)} samples")
print(f"Total Random Seeds: {len(seeds)}")

## 3. Validation Preloading - Fusion

In [ ]:
# -------------------------------------------------------------------------
# Preload validation datasets into RAM to avoid redundant disk I/O (Fusion)
# -------------------------------------------------------------------------
print("Preloading multimodal validation datasets (RP + AZ) into RAM...")
cached_val_tensors = {}

for vp in val_paths:
    # Use isolated test split for T197 to eliminate data leakage
    if "T197" in vp:
        temp_ds = test_ds_t197
    else:
        temp_ds = PKLDataset(folder_path=vp, modality="fusion")
        
    temp_loader = DataLoader(temp_ds, batch_size=256, shuffle=False)
    
    all_inputs, all_targets = [], []
    for x, y in temp_loader:
        all_inputs.append(x)
        all_targets.append(y)
        
    cached_val_tensors[vp] = {
        "inputs": torch.cat(all_inputs, dim=0),
        "targets": torch.cat(all_targets, dim=0)
    }

print("Multimodal validation datasets successfully preloaded into RAM.")

## 4. Training and Evaluation - Fusion

In [ ]:
# -------------------------------------------------------------------------
# Evaluation helper using in-memory cached tensors
# -------------------------------------------------------------------------
def fast_eval(model, inputs_tensor, targets_tensor, batch_size=256, device="cuda"):
    model.eval()
    correct = 0
    total = targets_tensor.size(0)
    with torch.no_grad():
        for i in range(0, total, batch_size):
            x = inputs_tensor[i:i + batch_size].to(device)
            y = targets_tensor[i:i + batch_size].to(device)
            preds = model(x).argmax(dim=1)
            targets_idx = y.argmax(dim=1)
            correct += (preds == targets_idx).sum().item()
    return 100.0 * correct / total

# -------------------------------------------------------------------------
# Phase 1 & 2: Pretraining on HC and Fine-Tuning across Transfer Sets (Fusion)
# -------------------------------------------------------------------------

for seed in seeds:
    print(f"\n==========================================")
    print(f">>> Running Multimodal Fusion Experiment for Seed: {seed}")
    print(f"==========================================")
    set_seed(seed)

    # ---------------------------------------------------------------------
    # Phase 1: Pretraining on HC source dataset (Multimodal Fusion)
    # ---------------------------------------------------------------------
    print("--> Phase 1: Pretraining on HC (T197 / Fusion)...")
    train_ds_hc, val_ds_hc = PKLDataset.split_dataset(
        folder_path=train_path_1,
        modality="fusion",
        train_ratio=0.8,
        random_seed=seed
    )
    train_loader_hc = DataLoader(train_ds_hc, batch_size=64, shuffle=True)

    # Initialize model with 2 input channels (RP + AZ)
    base_model = get_model(input_channels=2).to(device)
    opt_hc = optim.Adam(base_model.parameters(), lr=1e-3, weight_decay=1e-5)
    sch_hc = optim.lr_scheduler.StepLR(opt_hc, step_size=50, gamma=0.1)

    base_model = train_model(
        model=base_model,
        loader=train_loader_hc,
        criterion=criterion,
        optimizer=opt_hc,
        scheduler=sch_hc,
        num_epochs=10,
        device=device
    )

    # Save frozen weights from HC source domain
    pretrained_state = base_model.state_dict()

    # ---------------------------------------------------------------------
    # Phase 2: Transfer Learning (Fine-Tuning on DC Multimodal subsets)
    # ---------------------------------------------------------------------
    for t_name, n_samples in zip(transfer_sets, transfer_sample_sizes):
        print(f"\n--- Fine-tuning on Transfer Set: {t_name} ({n_samples} samples) ---")
        
        # Extract class-balanced subset strictly from the 70% multimodal training pool
        transfer_subset = get_balanced_transfer_subset(
            dataset=train_pool_dc,
            total_samples=n_samples,
            seed=seed
        )
        batch_sz = min(32, n_samples)
        loader_t = DataLoader(transfer_subset, batch_size=batch_sz, shuffle=True)

        # Initialize new model with 2 input channels and load HC pretrained weights
        tl_model = get_model(input_channels=2).to(device)
        tl_model.load_state_dict(pretrained_state)

        opt_dc = optim.Adam(tl_model.parameters(), lr=1e-3, weight_decay=1e-5)
        sch_dc = optim.lr_scheduler.StepLR(opt_dc, step_size=25, gamma=0.1)

        # Fine-tune on DC samples for 100 epochs
        tl_model = train_model(
            model=tl_model,
            loader=loader_t,
            criterion=criterion,
            optimizer=opt_dc,
            scheduler=sch_dc,
            num_epochs=100,
            device=device
        )

        # -----------------------------------------------------------------
        # Fast Evaluation across all held-out DC sheet thicknesses from RAM
        # -----------------------------------------------------------------
        for vp in val_paths:
            acc = fast_eval(
                model=tl_model,
                inputs_tensor=cached_val_tensors[vp]["inputs"],
                targets_tensor=cached_val_tensors[vp]["targets"],
                device=device
            )
            results[t_name][vp].append(acc)
            print(f"[Seed {seed}] {t_name} -> {vp}: {acc:.2f}%")

## 5. Summary of Results - Fusion

In [10]:
from pathlib import Path

summary_data = []

for t_name in transfer_sets:
    row = {"Transfer Set": t_name}
    for vp in val_paths:
        accs = results[t_name][vp]
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)
        
        # Robust cross-platform thickness extraction (e.g., 'T185')
        col_name = Path(vp).parent.name
        row[col_name] = f"{mean_acc:.2f} ± {std_acc:.2f}"
    summary_data.append(row)

# Construct summary DataFrame for Multimodal Fusion (RP + AZ)
df_fusion = pd.DataFrame(summary_data)

# Export consolidated fusion metrics to CSV
output_csv = "wear_Fusion_summary.csv"
df_fusion.to_csv(output_csv, index=False)

print("Consolidated Multimodal Fusion (RP + AZ) results:")
display(df_fusion)
print(f"Results successfully saved to: {output_csv}")

Consolidated Multimodal Fusion (RP + AZ) results:


,Transfer Set,T185,T188,T191,T194,T197
0,train_20,99.76 ± 0.28,99.78 ± 0.35,99.92 ± 0.10,99.41 ± 0.87,99.51 ± 0.49
1,train_50,99.28 ± 0.51,99.69 ± 0.36,99.95 ± 0.05,99.79 ± 0.32,99.84 ± 0.25
2,train_100,99.37 ± 0.90,99.86 ± 0.13,99.95 ± 0.04,99.91 ± 0.11,99.99 ± 0.04
3,train_200,99.36 ± 0.64,99.85 ± 0.24,99.98 ± 0.03,99.95 ± 0.08,100.00 ± 0.00
4,train_300,99.44 ± 0.21,99.89 ± 0.06,99.97 ± 0.03,99.96 ± 0.05,100.00 ± 0.00
5,train_400,99.25 ± 0.43,99.80 ± 0.20,99.96 ± 0.04,99.99 ± 0.02,100.00 ± 0.00
6,train_500,98.99 ± 0.58,99.89 ± 0.07,99.95 ± 0.03,99.99 ± 0.02,99.99 ± 0.02


Results successfully saved to: wear_Fusion_summary.csv
